In [18]:
import pandas as pd

In [19]:
df = pd.read_csv("IMDB Dataset.csv")

### Data Preprocessing

In [20]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [21]:
df.drop_duplicates(inplace=True)

In [22]:
## Converting to lowercase

df["review"] = df["review"].str.lower()

In [23]:
## Removing the URL

import re

def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text #(pattern,repl,string)

df["review"] = df["review"].apply(remove_urls)

In [24]:
## Remove the Puncutation

def remove_puncutations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"] = df["review"].apply(remove_puncutations)

In [25]:
## Removing HTML Tag 

def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_puncutations)

In [26]:
## removing the Stop words

import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [27]:
## Stemming

from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words =[]
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [28]:
## Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [29]:
y = df["sentiment"]

In [30]:
## Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=8000)
X = tf.fit_transform(df["review"])


### Data Set and DataLoader

In [31]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [32]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [33]:
X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [34]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

### Build RNN

In [38]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN Layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #Fully connect layer
        self.fc = nn.Linear(hidden_size,1)


    def forward(self,x):
        #optional
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _ = self.rnn(x,h0)
        # 1st value = hidden state of all the timesteps  =>(batch,seq_len,hidden_size)
        # 2nd value = Final hidden state of last timesteps

        out = self.fc(out[:,-1,:])
        return out

In [40]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())
